# Agent 5 — From Pages to Claims

A claim is a statement specific enough that a source could prove it wrong,
stored with its source and the **exact quote** that backs it. The model
extracts; the quote-audit checks it didn't invent.

**No class API key?** The precomputed claim table is real-shaped output;
the audit — the important part — runs offline on it.

In [ ]:
# The mini-web: nine pages about the (fictional) Riverside Community Garden.
# Small enough to read whole, real enough to research. One page is wrong on purpose.
MINIWEB = {
 "riverside-garden.org/about": {"date": "2026-05-10", "title": "Our garden today",
  "text": "The Riverside Community Garden has 60 plots and 48 member families. "
          "We grow vegetables for members and donate surplus to the food pantry."},
 "riverside-garden.org/history": {"date": "2023-05-02", "title": "Our history",
  "text": "Founded in 2019 with a dozen beds. The sign by the gate lists 48 plots, "
          "painted when we finished the 2023 season."},
 "riverside-garden.org/join": {"date": "2026-06-01", "title": "Join us",
  "text": "Want a plot? The waitlist currently holds 22 families. Members pay a "
          "small annual fee and share watering duties."},
 "lakeview-news.com/garden-expands": {"date": "2026-04-20", "title": "Garden adds 12 plots",
  "text": "The Riverside Community Garden completed its expansion this spring, "
          "taking the garden from 48 plots to 60. Organizers credit a city grant."},
 "lakeview-news.com/roundup-2023": {"date": "2023-09-15", "title": "Community roundup",
  "text": "At the Riverside garden, 31 member families closed out the 2023 season "
          "with a harvest festival."},
 "cityparks.gov/report-2026": {"date": "2026-03-14", "title": "Community garden census",
  "text": "Riverside Community Garden: 60 plots, 48 member families, established "
          "2019. Census conducted March 2026."},
 "cityparks.gov/grants-2025": {"date": "2025-11-08", "title": "2025 grant awards",
  "text": "Riverside Community Garden: $15,000 for expansion. The site's land "
          "lease with the parks department runs through 2028."},
 "gardenblog.example.com/visit": {"date": "2026-02-02", "title": "A visit to Riverside",
  "text": "Lovely afternoon at Riverside! I heard they have 600 plots now, which "
          "explains the crowds. The tomatoes were spectacular."},
 "gardenblog.example.com/opinion": {"date": "2026-01-05", "title": "Why gardens matter",
  "text": "Community gardens are the beating heart of a neighborhood. Riverside "
          "is a treasure and everyone loves it."},
}

import re as _re, collections as _c
def _words(text):
    return set(w for w in _re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2)
_DF = _c.Counter()                       # in how many pages does each word appear?
for _p in MINIWEB.values():
    for _w in _words(_p["title"] + " " + _p["text"]):
        _DF[_w] += 1

def search(query):
    """Score pages by shared words, each weighted by rarity (1/pages-containing-it).
    'riverside' is on every page and says nothing; 'waitlist' is on one and says a lot."""
    qwords = _words(query)
    scored = []
    for url, page in MINIWEB.items():
        shared = qwords & _words(page["title"] + " " + page["text"])
        scored.append((sum(1.0 / _DF[w] for w in shared), url, page["title"]))
    scored.sort(reverse=True)
    return [(url, title) for score, url, title in scored[:3] if score > 0.3]

def fetch(url):
    """Return a page's text with its receipt (url and date) attached."""
    page = MINIWEB[url]
    return {"url": url, "date": page["date"], "text": page["text"]}

print(f"{len(MINIWEB)} pages online.")
print("search('riverside garden plots') ->")
for url, title in search("riverside garden plots"):
    print("  ", url, "-", title)

In [ ]:
%pip install -q anthropic

In [ ]:
import os, getpass
# Ask your teacher for the class API key. It is never typed into a cell,
# never saved in the notebook - getpass keeps it out of your file.
try:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Class API key: ")
    HAVE_KEY = len(os.environ["ANTHROPIC_API_KEY"]) > 10
except Exception:
    HAVE_KEY = False
print("Key loaded." if HAVE_KEY else "No key - the notebook still teaches: precomputed outputs are shown below each live cell.")

In [ ]:
MODEL = "claude-opus-5"

def ask(prompt, system=None, max_tokens=1000):
    """One model call, plain text in and out."""
    import anthropic
    client = anthropic.Anthropic()
    kwargs = dict(model=MODEL, max_tokens=max_tokens,
                  messages=[{"role": "user", "content": prompt}])
    if system:
        kwargs["system"] = system
    return client.messages.create(**kwargs).content[-1].text

def get_json(prompt, tries=3):
    """Ask for JSON only; parse; re-ask on failure. The retry pattern from Build with LLMs."""
    import json as _json
    for attempt in range(tries):
        text = ask(prompt + "\n\nReply with ONLY valid JSON.")
        try:
            start = text.index("[") if "[" in text.split("{")[0] else text.index("{")
            return _json.loads(text[start:])
        except (ValueError, KeyError):
            continue
    raise RuntimeError("no valid JSON after retries")

## The claim schema

Same contract shape as Document Pipelines' schema prompts: exact keys,
null for anything not stated, JSON only. Aimed at sentences instead of
form fields.

In [ ]:
CLAIM_PROMPT = """From the page text below, extract every checkable factual claim
about the Riverside garden as JSON: a list of objects with keys
  claim   - one falsifiable statement (numbers and dates kept exact)
  quote   - the EXACT text from the page that backs it (copy, don't paraphrase)
Use null for anything the page does not literally state.

PAGE ({url}, {date}):
{text}"""

PRECOMPUTED_CLAIMS = [
 {"claim": "The garden has 60 plots (2026)", "quote": "The Riverside Community Garden has 60 plots",
  "source": "riverside-garden.org/about", "date": "2026-05-10"},
 {"claim": "The garden has 48 member families (2026)", "quote": "48 member families",
  "source": "riverside-garden.org/about", "date": "2026-05-10"},
 {"claim": "The sign lists 48 plots as of 2023", "quote": "The sign by the gate lists 48 plots",
  "source": "riverside-garden.org/history", "date": "2023-05-02"},
 {"claim": "The waitlist holds 22 families", "quote": "The waitlist currently holds 22 families",
  "source": "riverside-garden.org/join", "date": "2026-06-01"},
 {"claim": "The garden went from 48 plots to 60", "quote": "taking the garden from 48 plots to 60",
  "source": "lakeview-news.com/garden-expands", "date": "2026-04-20"},
 {"claim": "31 member families in 2023", "quote": "31 member families closed out the 2023 season",
  "source": "lakeview-news.com/roundup-2023", "date": "2023-09-15"},
 {"claim": "Census: 60 plots and 48 member families", "quote": "60 plots, 48 member families",
  "source": "cityparks.gov/report-2026", "date": "2026-03-14"},
 {"claim": "A $15,000 expansion grant was awarded in 2025", "quote": "$15,000 for expansion",
  "source": "cityparks.gov/grants-2025", "date": "2025-11-08"},
 {"claim": "The land lease runs through 2028", "quote": "land lease with the parks department runs through 2028",
  "source": "cityparks.gov/grants-2025", "date": "2025-11-08"},
 {"claim": "The garden has 600 plots", "quote": "I heard they have 600 plots now",
  "source": "gardenblog.example.com/visit", "date": "2026-02-02"},
 # the seeded stretch: the claim says more than its quote
 {"claim": "The grant doubled the garden's budget in 2025", "quote": "$15,000 for expansion",
  "source": "cityparks.gov/grants-2025", "date": "2025-11-08"},
]

if HAVE_KEY:
    claims = []
    for url, page in MINIWEB.items():
        rows = get_json(CLAIM_PROMPT.format(url=url, date=page["date"], text=page["text"]))
        for r in rows:
            r["source"], r["date"] = url, page["date"]
            claims.append(r)
else:
    claims = PRECOMPUTED_CLAIMS

print(f"{len(claims)} claims extracted")
for c in claims[:4]:
    print(f"  {c['claim']!r}  <- \"{c['quote']}\" [{c['source']}]")

## The quote audit — catch the stretch

Two mechanical checks per claim. First: is the quote really on the page?
(A fabricated quote is the worst failure.) Second, the human check: does
the quote actually SAY what the claim says? One claim in the table
stretches — the quote mentions a grant amount; the claim invents what the
grant did to a budget no page ever mentions.

In [ ]:
print("Check 1 - is every quote really on its page?")
for c in claims:
    on_page = c["quote"] in MINIWEB[c["source"]]["text"]
    print(("  ok      " if on_page else "  FABRICATED"), c["claim"])
    assert on_page, "a quote that is not on the page means the extractor invented it"

print()
print("Check 2 (by eye) - does the quote SAY what the claim says?")
suspect = [c for c in claims if "budget" in c["claim"].lower()]
for c in suspect:
    print(f"  claim: {c['claim']}")
    print(f"  quote: \"{c['quote']}\"")
    print("  The quote prices an expansion. The claim invents a budget effect")
    print("  no page states. Verbatim quote, stretched claim - mark it and")
    print("  it will not survive lesson 6.")

## Try it

1. Add a vibe to the claims list ("the garden is thriving") and write the
   quote you'd need. There isn't one — that's the point.
2. The blog's 600-plot claim passed the quote audit (the quote is real!).
   Write one sentence on why the audit alone can't kill it, and which
   lesson can.
3. **Build turn-in:** the full table marked backed / stretched / unbacked,
   plus a sentence on the stretch you found.